# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaifLatki/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row represents one anonymized page (`content_id`) for a client, captured as a recent content snapshot. The row is not a page-day series; it contains page-level aggregates for the most recent 90-day performance window and adjacent 30-day windows, but no explicit calendar date field is available.

In [4]:
import pandas as pd
from pathlib import Path

repo_root = Path.cwd()
for _ in range(6):
    candidate = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
    if candidate.exists():
        break
    repo_root = repo_root.parent
csv_path = candidate
assert csv_path.exists(), f"data/raw/content_refresh_anonymized.csv not found from cwd={Path.cwd()}"

df = pd.read_csv(csv_path)

window_cols = [c for c in df.columns if any(w in c for w in ("_90d", "_last_30d", "_prev_30d"))]
print("trend field missing values", df[["trend_direction", "trend_pct"]].isna().sum().to_dict())

print("window columns sample", sorted(window_cols)[:20])

print("rows", len(df))
print("window columns count", len(window_cols))

print("unique content_id", df["content_id"].nunique())
print("date-like columns", [c for c in df.columns if "date" in c.lower()])

print("duplicate content_id", df["content_id"].duplicated().sum())
print("unique client_id", df["client_id"].nunique())

trend field missing values {'trend_direction': 0, 'trend_pct': 3388}
window columns sample ['ai_sessions_90d', 'clicks_90d', 'clicks_last_30d', 'clicks_prev_30d', 'engaged_sessions_90d', 'impressions_90d', 'impressions_last_30d', 'impressions_prev_30d', 'pageviews_90d', 'scroll_events_90d', 'sessions_90d', 'sessions_last_30d', 'sessions_prev_30d', 'users_90d']
rows 30000
window columns count 14
unique content_id 30000
date-like columns ['days_since_last_update']
duplicate content_id 0
unique client_id 32


## 2. Fields: feature / label / context / excluded

Features are page-level search and engagement signals. The label is the observed traffic trend. Context are stable identifiers. Excluded fields are identifiers or non-predictive metadata that do not help the label directly.

In [5]:
import pandas as pd

df = pd.read_csv(csv_path)

feature_fields = [
    "search_volume", "competition", "competition_level", "cpc",
    "content_type", "main_intent", "word_count", "char_count",
    "provider_used", "model_used",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "age_tier", "age_tier_order", "days_since_last_update",
    "freshness_tier", "word_count_tier", "char_count_tier",
    "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "impression_tier", "position_tier",
    "trend_pct"
]
label_fields = ["trend_direction", "trend_pct"]
context_fields = ["content_id", "client_id"]
excluded_fields = ["content_id", "client_id"]

for name, fields in [
    ("features", feature_fields),
    ("label", label_fields),
    ("context", context_fields),
    ("excluded", excluded_fields),
]:
    present = [c for c in fields if c in df.columns]
    missing = [c for c in fields if c not in df.columns]
    print(f"{name}: present={len(present)}, missing={missing}")

print("\nlabel missing values")
print(df[label_fields].isna().sum().to_dict())
print("\ncontext uniqueness")
print(df[context_fields].nunique())
print("duplicate content_id rows", df["content_id"].duplicated().sum())

features: present=41, missing=[]
label: present=2, missing=[]
context: present=2, missing=[]
excluded: present=2, missing=[]

label missing values
{'trend_direction': 0, 'trend_pct': 3388}

context uniqueness
content_id    30000
client_id        32
dtype: int64
duplicate content_id rows 0


## 3. Verify it with queries (grain, counts, missing values, windows)

These checks confirm the row grain, the available windows, and the label completeness. They also demonstrate that the dataset is a page snapshot rather than a calendar series.

In [6]:
import pandas as pd

df = pd.read_csv(csv_path)

print("rows", len(df))
print("unique content_id", df["content_id"].nunique())
print("duplicate content_id", df["content_id"].duplicated().sum())
print("unique client_id", df["client_id"].nunique())
print("date-like columns", [c for c in df.columns if "date" in c.lower()])
window_cols = [c for c in df.columns if any(w in c for w in ("_90d", "_last_30d", "_prev_30d"))]
print("window columns count", len(window_cols))
print("window columns sample", sorted(window_cols)[:30])
print("\ntrend_direction distribution")
print(df["trend_direction"].value_counts(dropna=False).to_string())
print("\ntrend_pct summary")
print(df["trend_pct"].describe().to_string())
print("\nmissing values for trend fields")
print(df[["trend_direction", "trend_pct"]].isna().sum().to_dict())

rows 30000
unique content_id 30000
duplicate content_id 0
unique client_id 32
date-like columns ['days_since_last_update']
window columns count 14
window columns sample ['ai_sessions_90d', 'clicks_90d', 'clicks_last_30d', 'clicks_prev_30d', 'engaged_sessions_90d', 'impressions_90d', 'impressions_last_30d', 'impressions_prev_30d', 'pageviews_90d', 'scroll_events_90d', 'sessions_90d', 'sessions_last_30d', 'sessions_prev_30d', 'users_90d']

trend_direction distribution
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

trend_pct summary
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000

missing values for trend fields
{'trend_direction': 0, 'trend_pct': 3388}


## 4. Data limits

The data cannot tell us exact calendar dates or day-by-day seasonality because there is no explicit date field. It is a snapshot of recent page performance with 90-day and 30-day aggregates, so it cannot answer questions about daily timing, exact update dates, or external event causes. It also cannot prove that a trend is driven by a particular content change, only that the anonymized page's traffic direction has moved.

In [7]:
import pandas as pd

df = pd.read_csv(csv_path)

print("date-like columns", [c for c in df.columns if "date" in c.lower()])
print("total rows", len(df))
print("unique content_id", df["content_id"].nunique())
print("duplicate content_id", df["content_id"].duplicated().sum())
print("trend_direction unique", df["trend_direction"].nunique())
print("trend_pct min/max", df["trend_pct"].min(), df["trend_pct"].max())
print("\nexample snapshot rows")
print(df[["content_id", "impressions_90d", "clicks_90d", "sessions_90d", "trend_direction", "trend_pct"]].head(5).to_string(index=False))

date-like columns ['days_since_last_update']
total rows 30000
unique content_id 30000
duplicate content_id 0
trend_direction unique 5
trend_pct min/max -100.0 44900.0

example snapshot rows
          content_id  impressions_90d  clicks_90d  sessions_90d trend_direction  trend_pct
content_304f48230142             3803          29            17            down      -41.4
content_a1fb4e703a9e            15320           7             9            down      -57.7
content_9aa793d4d895            12581          11            11            down      -60.9
content_331d6c4de07b            11751          58            78          stable      -13.8
content_d99b7a2d90ca            19140          24           145            down      -34.7


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.